In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
path = os.path.join(path, "Q1_data.csv")
df = pd.read_csv(path)


In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df['Delivery_Time'].hist()

In [ ]:
df.columns

In [ ]:
df.shape

In [ ]:
df=df.drop(columns='Order_ID')

In [ ]:
df.shape #suppose decrease

In [ ]:

missing_values = df.isnull().sum()
print("Missing Values per Column:")
print(missing_values[missing_values > 0])

In [ ]:
# I want to check the types
df.info()

In [ ]:
missing_categorical=['Weather','Traffic_Level','Time_of_Day']
missing_numrical=['Courier_Experience_yrs','Delivery_Time']
for cl_c in missing_categorical:
  df[cl_c] = df[cl_c].fillna(df[cl_c].mode()[0])

for cl_n in missing_numrical:
  df[cl_n] = df[cl_n].fillna(df[cl_n].mean())

In [ ]:
missing_values = df.isnull().sum()
print("Missing Values per Column:")
print(missing_values[missing_values > 0])

In [ ]:
def check_duplicates(df):
  duplicates = df.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df)

In [ ]:
df.duplicated().sum()

In [ ]:
categorical_cols = df.select_dtypes(include=["object"]).columns
categorical_cols

In [ ]:
df[categorical_cols]

In [ ]:
from sklearn.preprocessing import LabelEncoder

for col in categorical_cols:
    print(f"Encoding column: {col}")
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])


In [ ]:
df[categorical_cols]

In [ ]:
#before scaling
df.describe()

In [ ]:
from sklearn.preprocessing import StandardScaler
all_featurs=df.columns.drop("Delivery_Time")
scaler = StandardScaler()
df[all_featurs]= scaler.fit_transform(df[all_featurs])

In [ ]:
#After scaling
df.describe()

In [ ]:
df.shape

In [ ]:
X=df.drop(columns='Delivery_Time')
print("X columns and shape: ",X.columns,' ',X.shape)#just to check
y=df['Delivery_Time']
print("y shape: ",y.shape)#just to check


In [ ]:
from sklearn.model_selection import KFold
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import  mean_absolute_error
n_splits = 5
kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
model=RandomForestRegressor(n_estimators=200)
lr_mae = []

In [ ]:
for fold_idx, (train_index, test_index) in enumerate(kf.split(X)):
  print(f"\nFold {fold_idx + 1}/{n_splits}")

  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

  # Train
  model.fit(X_train, y_train)

  # Predict
  y_pred = model.predict(X_test)

  # Calculate metrics
  mae = mean_absolute_error(y_test, y_pred)

    # Store results
  lr_mae.append(mae)


In [ ]:
print(f"  MAE:  {np.mean(lr_mae):.4f}")


In [ ]:
feature_importance = pd.DataFrame({
    'feature': X.columns,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(feature_importance['feature'], feature_importance['importance'])
plt.xlabel('Importance')
plt.title('Feature Importance')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
y_pred_plot = pd.DataFrame(y_pred)
y_pred_plot.hist()


In [ ]:
pip install catboost

In [ ]:
# Task Bonus: Write your code here:
from sklearn.ensemble import RandomForestRegressor
from catboost import CatBoostRegressor
models = {  "Random Forest Regressor": RandomForestRegressor(n_estimators=200),
  "CatBoost": CatBoostRegressor(verbose=0)
}

all_results = {}
for name in models:
  all_results[name] = {'mae': []}


In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)

for fold_idx, (train_index, test_index) in enumerate(kf.split(X)):
  print(f"\nFold {fold_idx + 1}/{n_splits}")

  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

  for model_name, model in models.items():
    print(f"Training {model_name}...")

    # Train
    model.fit(X_train, y_train)

    # Predict
    y_pred = model.predict(X_test)

    # Calculate metrics
    mae = mean_absolute_error(y_test, y_pred)

    # Store results
    all_results[model_name]["mae"].append(mae)

In [ ]:
 for model_name in all_results:
  print(f"\n{model_name}:")
  print(f"  MAE:  {np.mean(all_results[model_name]['mae']):.4f}")